In [ ]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

I renamed the files to have `_downloaded` suffix

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [5]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your OS.
2. Open a terminal and run:

```bash
ollama run llama3
```

This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

To check that the local server is running, you can also run:

```bash
curl http://localhost:11434
```

If you want to use it from Python, install the client with:

```bash
pip install ollama
```

Then you can call it with `ollama.chat(...)`.


## Asking meaningful questions without tools

In [6]:
answer = assistant.rag("How do I run Olama locally?")
print(answer)

The FAQ context doesn’t mention **Olama** specifically.

If you mean **running the course locally**, the relevant guidance is: yes, you can run it locally if you’re comfortable setting up **Python, `uv`, Jupyter, Docker, and any other tools needed for the module**. Also, if you run locally, **document your setup and keep your environment reproducible**.

If you meant a specific local setup for “Olama,” I’d need that exact name or more context from the FAQ.


In [7]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'You usually can, but it depends on whether the course is still open for enrollment.\n\nA few possibilities:\n- **If registration is still open:** you can likely enroll right away.\n- **If the course has already started:** you may still be able to join, depending on the instructor or school policy.\n- **If it’s full or closed:** you may need to waitlist or contact the organizer.\n\nIf you want, I can help you draft a quick message to the instructor or registration office asking whether you can still join.'

In [8]:
messages = [
    {"role": "user", "content": "How do I run Olama locally?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'I think you mean **Ollama**. Here’s the quick way to run it locally:\n\n## 1) Install Ollama\n- **macOS / Windows:** Download from https://ollama.com\n- **Linux:**\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\n## 2) Start the Ollama service\nUsually it starts automatically, but if needed:\n\n```bash\nollama serve\n```\n\n## 3) Download and run a model\nFor example, run Llama 3:\n\n```bash\nollama run llama3\n```\n\nThis will download the model the first time, then open an interactive chat.\n\n## 4) Use it from your app/API\nOllama exposes a local API by default at:\n\n```text\nhttp://localhost:11434\n```\n\nExample:\n```bash\ncurl http://localhost:11434/api/generate -d \'{\n  "model": "llama3",\n  "prompt": "Write a haiku about local AI."\n}\'\n```\n\n## 5) List available models\n```bash\nollama list\n```\n\n## 6) Stop/remove a model\n```bash\nollama rm llama3\n```\n\nIf you want, I can also show you:\n- how to run Ollama with **Docker**\n- how to call it from 

## Define the search function that will have an agentic element to it

In [9]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [10]:
# tell the model about the search tool, will be sending .json to OpenAI
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [11]:
# sending the question with the tool
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    # tools addition
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"How do I run Ollama locally"}', call_id='call_heUeAwtGCvrRQHsTmKoMYh48', name='search', type='function_call', id='fc_0da69622ffeedc4c006a3da2288948819e8eae73996f0ae13d', namespace=None, status='completed')]

One this that's interesting to note is that the query that was sent to OpenAI is not the originial query of , ""How do I run Olama locally?" but rather "un Ollama locally install start model pull command FAQ", which sound like gibberish to us, but makes more sense to machines

In [12]:
len(response.output)

1

In [13]:
# we know the arguments (from ResponseFunctionToolCall) and now we need to parse them
import json
call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
# turn the results back to json to send back to LLM
result_json = json.dumps(results, indent=2)

## the LLM said, "I want to invoke the search function with these arguments (query)"

In [14]:
args

{'query': 'How do I run Ollama locally'}

In [15]:
call.name

'search'

In [16]:
# result_json is what we need to send back to the LLM
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json
}

In [17]:
# append the function call output to messages, which is the message history
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [18]:
messages

[{'role': 'user', 'content': 'How do I run Olama locally?'},
 ResponseFunctionToolCall(arguments='{"query":"How do I run Ollama locally"}', call_id='call_heUeAwtGCvrRQHsTmKoMYh48', name='search', type='function_call', id='fc_0da69622ffeedc4c006a3da2288948819e8eae73996f0ae13d', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_heUeAwtGCvrRQHsTmKoMYh48',
  'output': '[\n  {\n    "id": "1d0b969028",\n    "course": "llm-zoomcamp",\n    "section": "Module 1: RAG",\n    "question": "Ollama: How to install Ollama?",\n    "answer": "First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\\n\\n- **macOS**: Download the `.pkg` and install it.\\n- **Windows**: Download the `.msi` and install it.\\n- **Linux**: Run the following command in the terminal:\\n\\n  ```bash\\n  curl -fsSL https://ollama.com/install.sh | sh\\n  ```\\n\\nOnce installed, open a terminal and type:\\n\\n```bash\\n

In [19]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    # now messages contains the previously sent use question and response
    input=messages,
    # tools addition
    tools=[search_tool],
)


In [20]:
print(response.output_text)

To run Ollama locally:

1. **Install Ollama**
   - macOS: download the `.pkg` from https://ollama.com/download
   - Windows: download the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model**
   ```bash
   ollama run llama3
   ```
   This will download the model and start a local chat interface.

3. **Check that the server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response showing available models.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(response['message']['content'])
   ```

If you get a connection error, restart the Ollama server:

```bash
ollama serve
```

If you want, I can also show you how to run a smaller model or how to use Ollama with Docker.


In [21]:
# But what happens when an LLM wants to make multiple calls instead of a singular call to tools? 